In [31]:
import pandas as pd

data_movies=pd.read_csv('ml-latest-small/movies.csv')
data_movies.head(5)



,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [32]:
data_raitings=pd.read_csv('ml-latest-small/ratings.csv')
C=data_raitings['rating'].mean()
print("Prosječna ocjena filmova u ovoj bazi je:",C)


Prosječna ocjena filmova u ovoj bazi je: 3.501556983616962


In [33]:
vote_counts=data_raitings.groupby('movieId')['rating'].count()
vote_counts.head(20)

movieId
1     215
2     110
3      52
4       7
5      49
6     102
7      54
8       8
9      16
10    132
11     70
12     19
13      8
14     18
15     13
16     82
17     67
18     20
19     88
20     15
Name: rating, dtype: int64

In [34]:
m=vote_counts.quantile(0.90)
print(f"Minimalan broj glasova potreban za listu je: {m}")

Minimalan broj glasova potreban za listu je: 27.0


In [35]:
data_raitings['vote_count']=data_raitings['movieId'].map(vote_counts)

In [36]:

q_movies = data_raitings.copy().loc[data_raitings['vote_count'] >= m]
print(q_movies.shape)


(60632, 5)


In [37]:
print(data_raitings.shape)

(100836, 5)


In [38]:
q_movies.info()

<class 'pandas.DataFrame'>
Index: 60632 entries, 0 to 100830
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   userId      60632 non-null  int64  
 1   movieId     60632 non-null  int64  
 2   rating      60632 non-null  float64
 3   timestamp   60632 non-null  int64  
 4   vote_count  60632 non-null  int64  
dtypes: float64(1), int64(4)
memory usage: 2.8 MB


In [39]:
movie_means=data_raitings.groupby('movieId')['rating'].mean()

In [42]:
q_movies['vote_average'] = q_movies['movieId'].map(movie_means)

In [40]:
def weighted_rating(x,m=m,C=C):
    v=x['vote_count']
    R=x['vote_average']
    return (v/(v+m)*R)+(m/(m+v)*C)

In [43]:
q_movies['score']=q_movies.apply(weighted_rating,axis=1)

In [45]:
final_scores = q_movies[['movieId', 'vote_count', 'vote_average', 'score']].drop_duplicates(subset='movieId')

In [47]:
movies_with_raiting=pd.merge(final_scores,data_movies[['movieId','title']],on='movieId')


In [49]:
top_movies=movies_with_raiting.sort_values('score',ascending=False)
top_movies[['title','vote_count','vote_average','score']].head(20)

,title,vote_count,vote_average,score
162,"Shawshank Redemption, The (1994)",317,4.429022,4.356227
528,"Godfather, The (1972)",192,4.289062,4.191973
135,Fight Club (1999),218,4.272936,4.187927
13,Star Wars: Episode IV - A New Hope (1977),251,4.231076,4.160223
4,"Usual Suspects, The (1995)",204,4.237745,4.151697
25,Schindler's List (1993),220,4.225000,4.145919
14,Pulp Fiction (1994),307,4.197068,4.140844
54,Star Wars: Episode V - The Empire Strikes Back...,211,4.215640,4.134630
117,"Matrix, The (1999)",278,4.192446,4.131285
604,"Godfather: Part II, The (1974)",129,4.259690,4.128475
